In [1]:
import sys
import os

import tensorflow as tf

os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"
sys.path.append(os.path.abspath(os.path.join(os.path.dirname('variational_ae.py'), '..')))
sys.path.append(os.path.abspath(os.path.join(os.path.dirname('encoder.py'), '..')))
sys.path.append(os.path.abspath(os.path.join(os.path.dirname('decoder.py'), '..')))

import numpy as np
from sklearn.model_selection import train_test_split
import h5py

from variational_ae import VariationalAutoencoder
from encoder import EncoderBuilder
from decoder import DecoderBuilder

2025-08-16 16:47:03.596297: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-16 16:47:05.048292: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-08-16 16:47:07.824474: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"         # Keeps GPU order consistent
os.environ["CUDA_VISIBLE_DEVICES"] = "0"               # Makes only GPU 0 visible (useful even with 1 GPU)

gpus = tf.config.experimental.list_physical_devices("GPU")
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)  # Prevents TF from using all GPU memory at once
    except RuntimeError as e:
        print("Error: ", e)
        exit(-1)

In [3]:
with h5py.File('Dataset/log_mel_spec_data_dataset.h5', 'r') as h5f:
    log_mel_spec_data_train = h5f['train'][:]
    log_melspec_data_labels = h5f['label'][:]

In [4]:
log_mel_spec_data_train = log_mel_spec_data_train[..., np.newaxis]

log_mel_spec_x_train, log_mel_spec_x_val = train_test_split(log_mel_spec_data_train, test_size=0.05, random_state=42)

In [5]:
print("Log Spec Train shape:", log_mel_spec_x_train.shape)
print("Log Spec Validation shape:", log_mel_spec_x_val.shape)

Log Spec Train shape: (28500, 128, 64, 1)
Log Spec Validation shape: (1500, 128, 64, 1)


In [6]:
LEARNING_RATE = 0.0005
BATCH_SIZE = 64
EPOCHS = 50

In [7]:
input_shape = log_mel_spec_x_train.shape[1:]
latent_space_dim = 128
decoder_out_filter = 1

In [8]:
# Hyperparameters for the Variational Autoencoder
recon_weight = 1.0  # Weight for the reconstruction loss.
beta = 1.0  # Weight for the KL divergence loss.

In [9]:
conv_layers_config=[
    {'filters': 512, 'kernel_size': (3, 3), 'strides': 2},
    {'filters': 256, 'kernel_size': (3, 3), 'strides': 2},
    {'filters': 128, 'kernel_size': (3, 3), 'strides': 2},
    {'filters': 64, 'kernel_size': (3, 3), 'strides': 2},
    {'filters': 32, 'kernel_size': (3, 3), 'strides': (2, 1)},
]

In [10]:
dummy_input = tf.random.normal((1, input_shape[0], input_shape[1], input_shape[2]))

I0000 00:00:1755334033.193449     920 gpu_process_state.cc:208] Using CUDA malloc Async allocator for GPU: 0
I0000 00:00:1755334033.194578     920 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9711 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060, pci bus id: 0000:01:00.0, compute capability: 8.6


In [11]:
encoder = EncoderBuilder(latent_space_dim, conv_layers_config)
encoder(dummy_input)
shape_before_bottleneck = encoder.get_shape_before_bottleneck()
encoder_model = encoder.build_graph(input_shape=dummy_input.shape)

2025-08-16 16:47:14.575671: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 90300


In [12]:
decoder = DecoderBuilder(shape_before_bottleneck, decoder_out_filter, conv_layers_config)
dummy_input = tf.random.normal((1, latent_space_dim))
decoder(dummy_input)
decoder_model = decoder.build_graph(input_shape=dummy_input.shape)

Number of Neurons:  <class 'numpy.int64'>
<class 'tuple'>
Value:  (4, 4, 32)
1st x's Shape:  TensorShape([1, 512])
2nd x's Reshape:  TensorShape([1, 4, 4, 32])
1st x's Shape:  (None, 512)
2nd x's Reshape:  (None, 4, 4, 32)


In [13]:
vae = VariationalAutoencoder(
        recon_weight=recon_weight,
        beta=beta,
        encoder=encoder_model,
        decoder=decoder_model
    )

In [14]:
dummy_input = tf.random.normal((1, input_shape[0], input_shape[1], input_shape[2]))
vae(dummy_input)

<tf.Tensor: shape=(1, 128, 64, 1), dtype=float32, numpy=
array([[[[0.500004  ],
         [0.49986526],
         [0.50031465],
         ...,
         [0.49992263],
         [0.5001525 ],
         [0.49991685]],

        [[0.5000371 ],
         [0.49980837],
         [0.50023323],
         ...,
         [0.49937043],
         [0.50032467],
         [0.49998927]],

        [[0.50042343],
         [0.4996762 ],
         [0.5001873 ],
         ...,
         [0.49900016],
         [0.49988165],
         [0.50014955]],

        ...,

        [[0.500062  ],
         [0.4994576 ],
         [0.49980578],
         ...,
         [0.4988951 ],
         [0.5004999 ],
         [0.50001365]],

        [[0.49997723],
         [0.49944404],
         [0.49954563],
         ...,
         [0.49924085],
         [0.4997281 ],
         [0.49995595]],

        [[0.5000358 ],
         [0.49983907],
         [0.49973324],
         ...,
         [0.49955687],
         [0.49887475],
         [0.49995565]]]], dtyp

In [15]:
from tensorflow.keras.optimizers import Adam
vae.compile(optimizer=Adam(learning_rate=0.0001))

In [16]:
vae.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 128, 64,   │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_conv_layer… │ (None, 64, 32,    │      4,608 │ input_layer[0][0] │
│ (Conv2D)            │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_batch_norm… │ (None, 64, 32,    │      2,048 │ encoder_conv_lay… │
│ (BatchNormalizatio… │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_leaky_relu… │ (None, 64, 32,    │          0 │ encoder_batch_no… │
│ (LeakyReLU)         │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_conv_layer… │ (None, 32, 16,    │  1,179,648 │ encoder_leaky_re… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_batch_norm… │ (None, 32, 16,    │      1,024 │ encoder_conv_lay… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_leaky_relu… │ (None, 32, 16,    │          0 │ encoder_batch_no… │
│ (LeakyReLU)         │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_conv_layer… │ (None, 16, 8,     │    294,912 │ encoder_leaky_re… │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_batch_norm… │ (None, 16, 8,     │        512 │ encoder_conv_lay… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_leaky_relu… │ (None, 16, 8,     │          0 │ encoder_batch_no… │
│ (LeakyReLU)         │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_conv_layer… │ (None, 8, 4, 64)  │     73,728 │ encoder_leaky_re… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_batch_norm… │ (None, 8, 4, 64)  │        256 │ encoder_conv_lay… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_leaky_relu… │ (None, 8, 4, 64)  │          0 │ encoder_batch_no… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_conv_layer… │ (None, 4, 4, 32)  │     18,432 │ encoder_leaky_re… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_batch_norm… │ (None, 4, 4, 32)  │        128 │ encoder_conv_lay… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_leaky_relu… │ (None, 4, 4, 32)  │          0 │ encoder_batch_no… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_flatten_la… │ (None, 512)       │          0 │ encoder_leaky_re

 Total params: 1,706,624 (6.51 MB)

 Trainable params: 1,704,640 (6.50 MB)

 Non-trainable params: 1,984 (7.75 KB)

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_dense_layer (Dense)     │ (None, 512)            │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_reshape_layer (Reshape) │ (None, 4, 4, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_conv_transpose_layer_3  │ (None, 8, 4, 32)       │         9,216 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_batch_norm_layer_3      │ (None, 8, 4, 32)       │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_relu_layer_3 (ReLU)     │ (None, 8, 4, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_conv_transpose_layer_2  │ (None, 16, 8, 64)      │        18,432 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_batch_norm_layer_2      │ (None, 16, 8, 64)      │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_relu_layer_2 (ReLU)     │ (None, 16, 8, 64)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_conv_transpose_layer_1  │ (None, 32, 16, 128)    │        73,728 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_batch_norm_layer_1      │ (None, 32, 16, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_relu_layer_1 (ReLU)     │ (None, 32, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_conv_transpose_layer_0  │ (None, 64, 32, 256)    │       294,912 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_batch_norm_layer_0      │ (None, 64, 32, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_relu_layer_0 (ReLU)     │ (None, 64, 32, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_conv_transpose_layer_5  │ (None, 128, 64, 1)     │         2,304 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_output_activation       │ (None, 128, 64, 1)     │             0 │
│ (Activation)                    │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 466,560 (1.78 MB)

 Trainable params: 465,600 (1.78 MB)

 Non-trainable params: 960 (3.75 KB)

In [17]:
history = vae.fit(
    x=log_mel_spec_x_train,
    y=log_mel_spec_x_train, # Autoencoders typically use the same data for input and output
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(log_mel_spec_x_val, log_mel_spec_x_val), # Validation data for monitoring
    shuffle=True
)

Epoch 1/50


2025-08-16 16:47:22.651037: I external/local_xla/xla/service/service.cc:163] XLA service 0x7d9d90006210 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-08-16 16:47:22.651074: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 3060, Compute Capability 8.6
2025-08-16 16:47:22.817819: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-08-16 16:47:23.628536: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-08-16 16:47:23.628615: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of th

446/446 ━━━━━━━━━━━━━━━━━━━━ 68s 106ms/step - kl_loss: 30.3943 - reconstruction_loss: 502.2157 - total_loss: 532.6099 - val_val_kl_loss: 15.8570 - val_val_reconstruction_loss: 279.9761 - val_val_total_loss: 295.8331
Epoch 2/50
446/446 ━━━━━━━━━━━━━━━━━━━━ 34s 75ms/step - kl_loss: 22.0558 - reconstruction_loss: 149.9330 - total_loss: 171.9888 - val_val_kl_loss: 22.0932 - val_val_reconstruction_loss: 137.8860 - val_val_total_loss: 159.9791
Epoch 3/50
446/446 ━━━━━━━━━━━━━━━━━━━━ 34s 76ms/step - kl_loss: 20.9753 - reconstruction_loss: 136.6265 - total_loss: 157.6018 - val_val_kl_loss: 22.5951 - val_val_reconstruction_loss: 130.1202 - val_val_total_loss: 152.7154
Epoch 4/50
446/446 ━━━━━━━━━━━━━━━━━━━━ 33s 75ms/step - kl_loss: 20.4014 - reconstruction_loss: 131.3897 - total_loss: 151.7912 - val_val_kl_loss: 20.6524 - val_val_reconstruction_loss: 126.0367 - val_val_total_loss: 146.6892
Epoch 5/50
446/446 ━━━━━━━━━━━━━━━━━━━━ 33s 75ms/step - kl_loss: 19.7651 - reconstruction_loss: 127.0711 -

In [18]:
vae.save("initial_full_vae_log_mel_spec.keras")
encoder, decoder = vae.get_models()
encoder.save("initial_encoder_vae_log_mel_spec.keras")
decoder.save("initial_decoder_vae_log_mel_spec.keras")